# Phase 4 CJPE — Experiment 6/6: Knowledge distillation (mandatory)

**Technique:** Knowledge distillation

**This is the experiment the phase makes mandatory.** The teacher is the fully fine-tuned Phase 1 model from notebook 01 (19 layers for the decoder, 14 for the encoder). The student is a shallower encoder built from an evenly spaced subset of the teacher's layers, so it inherits Phase 1 representations rather than starting cold. It learns from the teacher's softened logits plus the true labels.

**Run notebook 01 first** — this one loads the checkpoint it saves.

---
### Self-contained
No `.py` files, no setup step. The library cells in section 2 are identical in
all six notebooks, so any of them runs on its own. You only need your own
`a1_tokenizer.py`, `bpe_40k.json` and `best_enc.pt` in Drive.

### How the six fit together
Every notebook shares the same data pipeline, 512-token window, mean-pool +
linear head, training loop and evaluation. The **only** thing that differs is
the adaptation technique — which is what the phase requires ("these experiments
must not consist only of changing hyperparameters of the same method").

Results append to `cjpe_results.json` in Drive, so the comparison table at the
end of any notebook shows every experiment run so far.

**Order:** 01 first (it is the reference the others are compared against, and it
saves the teacher notebook 06 needs), then 02-05 in any order, 06 last.

Set `quick=True` in the config cell for a ~10 minute smoke run before committing
a full GPU hour.

## 1 — Setup

In [ ]:
# ---- Google Colab: mount Drive ----
from google.colab import drive
drive.mount("/content/drive")

import os, sys, glob, subprocess

# ---- EDIT: the Drive folder holding a1_tokenizer.py, bpe_40k.json, best_enc.pt ----
DRIVE = "/content/drive/MyDrive"

# ---- EDIT: your HuggingFace token (the dataset repo is gated) ----
HF_TOKEN = ""

# ---- which Phase 1 model to adapt ----
BACKBONE       = "encoder"      # "encoder" = best_enc.pt (MHA), "decoder" = best.pt (GQA)
DEC_CKPT_NAME  = "best.pt"      # the decoder checkpoint file name
MAX_LEN        = 512            # both Phase 1 models were pretrained at 512
ROPE_TABLE_LEN = 1024           # the decoder precomputes RoPE at 2 x max_seq_len
# -----------------------------------------------------------------------------

WORK          = DRIVE                       # outputs -> {DRIVE}/cs781_phase4
TOK_JSON_PATH = f"{DRIVE}/bpe_40k.json"
TOK_PY_PATH   = f"{DRIVE}/a1_tokenizer.py"
ENC_PATH      = f"{DRIVE}/best_enc.pt"
DEC_PATH      = f"{DRIVE}/{DEC_CKPT_NAME}"

need = {"bpe_40k.json": TOK_JSON_PATH, "a1_tokenizer.py": TOK_PY_PATH,
        ("best_enc.pt" if BACKBONE == "encoder" else DEC_CKPT_NAME):
            (ENC_PATH if BACKBONE == "encoder" else DEC_PATH)}
for f, pth in need.items():
    ok = os.path.exists(pth)
    print(("FOUND    " if ok else "MISSING  ") + f
          + (f"  {os.path.getsize(pth)/1e6:.1f} MB" if ok else ""))
missing = [f for f, pth in need.items() if not os.path.exists(pth)]
if missing:
    raise FileNotFoundError(f"missing {missing} in {DRIVE} — fix DRIVE above")

if DRIVE not in sys.path:
    sys.path.insert(0, DRIVE)               # so `import a1_tokenizer` finds yours

try:                                        # preinstalled on Colab
    import huggingface_hub, sklearn
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install",
                           "huggingface_hub", "scikit-learn"])

import torch
print(f"\noutputs -> {WORK}/cs781_phase4")
print("torch", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          f"({torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB)")
else:
    print("WARNING: no GPU — Runtime > Change runtime type > T4 GPU")

Mounted at /content/drive
FOUND    bpe_40k.json  1.2 MB
FOUND    a1_tokenizer.py  0.0 MB
FOUND    best_enc.pt  1565.3 MB

outputs -> /content/drive/MyDrive/cs781_phase4
torch 2.11.0+cu130
GPU: NVIDIA L4 (24 GB)


## 2 — Library

Everything the experiment needs, defined inline. These six cells are **identical
in all six notebooks**, so whichever one you open is self-contained — there are
no `.py` files to upload and no setup notebook to run first.

Run them once per session (Runtime → Run all does it for you). You can collapse
them; nothing below depends on reading them.

The only file this still needs from your Drive is **`a1_tokenizer.py`**, your own
Phase 1 tokenizer module — it has to be yours so it matches `bpe_40k.json`
exactly.

### Library 1/6 — imports, configuration, tokenizer

In [ ]:
import json, math, os, random, re, sys, time
from dataclasses import dataclass, field, asdict
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

@dataclass
class CJPEConfig:
    # --- paths (override in the notebook) ---
    drive_dir: str = "/content/drive/MyDrive"
    enc_ckpt: str = "/content/drive/MyDrive/best_enc.pt"
    dec_ckpt: str = "/content/drive/MyDrive/best.pt"
    tokenizer_json: str = "/content/drive/MyDrive/bpe_40k.json"
    cache_dir: str = "/content/drive/MyDrive/cjpe_cache"
    results_json: str = "/content/drive/MyDrive/cjpe_results.json"

    # --- document representation (held CONSTANT across the 6 experiments) ---
    max_len: int = 512
    window: str = "head_tail"        # head | tail | head_tail
    head_frac: float = 0.5           # for head_tail: share of max_len from the start

    # --- backbone ---
    n_heads: int = 12
    n_query_heads: int = 12       # decoder: query heads (kv inferred)
    rope_table_len: int | None = None   # None -> 2 * max_len
    dec_pool: str = "last"        # decoder readout: last | mean
    backbone: str = "encoder"     # encoder | decoder
    dropout: float = 0.1
    rope_theta: float = 10000.0

    # --- training ---
    lr: float = 2e-5                 # backbone lr (PEFT notebooks raise this)
    head_lr: float = 1e-3            # classifier head always trains faster
    weight_decay: float = 0.01
    epochs: int = 2
    batch_size: int = 8
    grad_accum: int = 2
    warmup_frac: float = 0.06
    max_grad_norm: float = 1.0
    label_smoothing: float = 0.0
    seed: int = 42
    eval_every: int = 400            # optimizer steps between dev evaluations
    patience: int = 4                # dev evals without macro-F1 improvement

    # --- data size (set quick=True for a fast smoke run) ---
    quick: bool = False
    max_train: int | None = None     # None = use all of train.json
    max_dev: int | None = None

    hf_repo: str = "Exploration-Lab/CS-781-Capstone"
    hf_token: str | None = None

    def __post_init__(self):
        if self.quick:
            self.max_train = self.max_train or 3000
            self.max_dev = self.max_dev or 800
            self.epochs = 1
            self.eval_every = 100


def set_seed(s: int):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PAD_NAMES = ["<pad>", "[PAD]", "<PAD>", "[pad]"]


def load_tokenizer(cfg: CJPEConfig, py_dir: str | None = None):
    """Returns (tokenizer, pad_id, base_vocab_size)."""
    d = py_dir or str(Path(cfg.tokenizer_json).parent)
    if d not in sys.path:
        sys.path.insert(0, d)
    from a1_tokenizer import BPETokenizer
    tok = BPETokenizer.load(cfg.tokenizer_json)
    base = int(tok.vocab_size)

    pad = None
    st = getattr(tok, "special_tokens", None)
    if isinstance(st, dict):
        for n in PAD_NAMES:
            if isinstance(st.get(n), int):
                pad = int(st[n]); break
    if pad is None:
        for attr in ("token_to_id", "vocab", "encoder", "stoi"):
            v = getattr(tok, attr, None)
            if isinstance(v, dict):
                for n in PAD_NAMES:
                    if isinstance(v.get(n), int):
                        pad = int(v[n]); break
            if pad is not None:
                break
    if pad is None:
        raise ValueError("Could not resolve the pad id from this tokenizer.")
    return tok, pad, base

### Library 2/6 — data loading, windowing, caching

In [ ]:
def _hf_path(cfg: CJPEConfig, rel: str) -> str:
    from huggingface_hub import hf_hub_download
    return hf_hub_download(cfg.hf_repo, rel, repo_type="dataset", token=cfg.hf_token)


def load_split(cfg: CJPEConfig, split: str, local: str | None = None):
    """split: train | dev | test. Returns list of (id, text, label|None)."""
    if local and os.path.exists(local):
        path = local
    else:
        sub = "Test" if split == "test" else "Train"
        path = _hf_path(cfg, f"Phase_4_CJPE/{sub}/{split}.json")
    with open(path) as f:
        raw = json.load(f)
    rows = raw.values() if isinstance(raw, dict) else raw
    out = []
    for r in rows:
        if not isinstance(r, dict):
            continue
        txt = r.get("text") or r.get("document") or ""
        if isinstance(txt, list):
            txt = " ".join(map(str, txt))
        if not txt:
            continue
        lab = r.get("label", r.get("labels"))
        if isinstance(lab, list):
            lab = lab[0] if lab else None
        out.append((str(r.get("id", len(out))), txt,
                    None if lab is None else int(lab)))
    return out


# ---- windowing: how a long judgment becomes max_len tokens ----

def encode_window(tok, text, pad_id, cfg: CJPEConfig):
    """Returns (ids[max_len], n_real). Pre-truncates by words so the pure-Python
    BPE never processes material that would be discarded: encode() splits on
    whitespace and encodes each word independently, so the ids for the first N
    words are a prefix of the full document's ids (and likewise for a suffix)."""
    L, w = cfg.max_len, text.split()
    budget_words = 6 * L                      # ~4x margin over tokens-per-word

    if cfg.window == "head":
        ids = tok.encode(" ".join(w[:budget_words]), add_special_tokens=True)[:L]
    elif cfg.window == "tail":
        ids = tok.encode(" ".join(w[-budget_words:]), add_special_tokens=True)[-L:]
    elif cfg.window == "head_tail":
        n_head = int(round(L * cfg.head_frac))
        n_tail = L - n_head
        h_full = tok.encode(" ".join(w[:budget_words]), add_special_tokens=True)
        # A document that already fits must NOT be spliced, or the head and the
        # tail slice would both cover it and the model would see it twice.
        if len(w) <= budget_words and len(h_full) <= L:
            ids = h_full
        else:
            t = tok.encode(" ".join(w[-budget_words:]), add_special_tokens=True)
            ids = h_full[:n_head] + t[-n_tail:]
    else:
        raise ValueError(f"unknown window {cfg.window!r}")

    n = max(len(ids), 1)
    if len(ids) < L:
        ids = ids + [pad_id] * (L - len(ids))
    return ids[:L], min(n, L)


def encode_split(tok, rows, pad_id, cfg: CJPEConfig, desc=""):
    X = np.full((len(rows), cfg.max_len), pad_id, dtype=np.int32)
    n = np.zeros(len(rows), dtype=np.int32)
    t0 = time.time()
    for i, (_, text, _) in enumerate(rows):
        ids, k = encode_window(tok, text, pad_id, cfg)
        X[i] = ids; n[i] = k
        if desc and i % 4000 == 0 and i:
            print(f"  {desc} {i}/{len(rows)} ({time.time()-t0:.0f}s)")
    if desc:
        print(f"  {desc} {len(rows)} docs in {time.time()-t0:.0f}s")
    return X, n


def prepare_data(cfg: CJPEConfig, tok, pad_id, splits=("train", "dev"),
                 local_paths: dict | None = None):
    """Tokenizes once and caches to Drive, keyed by window settings, so the
    other five notebooks reuse it instead of re-tokenizing."""
    local_paths = local_paths or {}
    Path(cfg.cache_dir).mkdir(parents=True, exist_ok=True)
    key = f"{cfg.window}_{cfg.max_len}_{cfg.head_frac:.2f}"
    out = {}
    for sp in splits:
        cap = cfg.max_train if sp == "train" else cfg.max_dev
        tag = f"{sp}_{key}" + (f"_n{cap}" if cap else "")
        fx = Path(cfg.cache_dir) / f"X_{tag}.npy"
        fn = Path(cfg.cache_dir) / f"n_{tag}.npy"
        fm = Path(cfg.cache_dir) / f"meta_{tag}.json"
        if fx.exists() and fn.exists() and fm.exists():
            meta = json.loads(fm.read_text())
            out[sp] = (np.load(fx), np.load(fn),
                       np.array(meta["y"]) if meta["y"] is not None else None,
                       meta["ids"])
            print(f"[cache hit ] {sp}: {out[sp][0].shape[0]} docs  ({fx.name})")
            continue
        rows = load_split(cfg, sp, local_paths.get(sp))
        if cap and len(rows) > cap:
            rng = np.random.default_rng(cfg.seed)
            sel = rng.choice(len(rows), cap, replace=False)
            rows = [rows[i] for i in sel]
        print(f"[tokenizing] {sp}: {len(rows)} docs, window={cfg.window}")
        X, n = encode_split(tok, rows, pad_id, cfg, desc=sp)
        y = None if rows[0][2] is None else np.array([r[2] for r in rows], dtype=np.int64)
        ids = [r[0] for r in rows]
        np.save(fx, X); np.save(fn, n)
        fm.write_text(json.dumps({"y": None if y is None else y.tolist(), "ids": ids}))
        out[sp] = (X, n, y, ids)
    return out


class CJPEDataset(torch.utils.data.Dataset):
    def __init__(self, X, n, y=None):
        self.X, self.n, self.y = X, n, y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        item = (torch.from_numpy(self.X[i].astype(np.int64)), int(self.n[i]))
        return item + ((int(self.y[i]),) if self.y is not None else (-1,))


def make_loader(X, n, y, batch_size, shuffle, seed=42):
    g = torch.Generator(); g.manual_seed(seed)
    return torch.utils.data.DataLoader(
        CJPEDataset(X, n, y), batch_size=batch_size, shuffle=shuffle,
        num_workers=2, pin_memory=(DEVICE.type == "cuda"), generator=g,
        drop_last=False)

### Library 3/6 — the Phase 1 backbones (encoder + decoder) and classifier

In [ ]:
# Copied from encoder_legal_lm_fixed.ipynb so the Phase 1 state_dict loads
# into exactly the code that produced it.

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        rms = torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return (x.float() * rms).to(x.dtype) * self.weight


def precompute_rope(dim, max_len, theta=10000.0, device=None):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2, device=device).float() / dim))
    t = torch.arange(max_len, device=device).float()
    ang = torch.outer(t, freqs)
    return ang.cos(), ang.sin()


def apply_rope(x, cos, sin):
    T = x.size(2)
    c = cos[:T].unsqueeze(0).unsqueeze(0)
    s = sin[:T].unsqueeze(0).unsqueeze(0)
    hd = x.size(-1) // 2
    x1, x2 = x[..., :hd], x[..., hd:]
    rot = torch.cat([-x2, x1], dim=-1)
    c_e = torch.cat([c.expand(-1, -1, -1, hd), c.expand(-1, -1, -1, hd)], -1)
    s_e = torch.cat([s.expand(-1, -1, -1, hd), s.expand(-1, -1, -1, hd)], -1)
    return x * c_e + rot * s_e


class MultiHeadSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.nh, self.hd, self.dm = cfg.n_heads, cfg.head_dim, cfg.d_model
        self.q = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.k = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.v = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.o = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.attn_drop = nn.Dropout(cfg.dropout)
        self.resid_drop = nn.Dropout(cfg.dropout)
        self.scale = 1.0 / math.sqrt(cfg.head_dim)
    def forward(self, x, cos, sin, padding_mask=None):
        B, T, _ = x.shape
        q = self.q(x).view(B, T, self.nh, self.hd).transpose(1, 2)
        k = self.k(x).view(B, T, self.nh, self.hd).transpose(1, 2)
        v = self.v(x).view(B, T, self.nh, self.hd).transpose(1, 2)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        w = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        if padding_mask is not None:
            w = w.masked_fill(padding_mask.unsqueeze(1).unsqueeze(2), float("-inf"))
        w = F.softmax(w, dim=-1, dtype=torch.float32).to(q.dtype)
        w = self.attn_drop(w)
        out = torch.matmul(w, v).transpose(1, 2).contiguous().view(B, T, self.dm)
        return self.resid_drop(self.o(out))


class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.gate = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.up   = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.down = nn.Linear(cfg.d_ff, cfg.d_model, bias=False)
        self.drop = nn.Dropout(cfg.dropout)
    def forward(self, x):
        return self.drop(self.down(F.silu(self.gate(x)) * self.up(x)))


class EncoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model); self.attn = MultiHeadSelfAttention(cfg)
        self.norm2 = RMSNorm(cfg.d_model); self.ffn  = SwiGLU(cfg)
    def forward(self, x, cos, sin, padding_mask=None):
        x = x + self.attn(self.norm1(x), cos, sin, padding_mask)
        return x + self.ffn(self.norm2(x))


class BackboneCfg:
    pass


class LegalEncoder(nn.Module):
    """Phase 1 encoder without the MLM head."""
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        self.layers = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg.n_layers)])
        self.norm = RMSNorm(cfg.d_model)
        cos, sin = precompute_rope(cfg.head_dim, cfg.max_seq_len, cfg.rope_theta)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        self.apply(self._init)   # only matters for tensors absent from the ckpt

    @staticmethod
    def _init(m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)

    def forward(self, ids, padding_mask=None):
        T = ids.size(1)
        if T > self.rope_cos.size(0):
            raise ValueError(
                f"sequence length {T} exceeds the RoPE table ({self.rope_cos.size(0)}). "
                f"Phase 1 precomputed RoPE at max_seq_len only; longer context needs "
                f"the buffers rebuilt and position interpolation.")
        x = self.drop(self.tok_emb(ids))
        for layer in self.layers:
            x = layer(x, self.rope_cos, self.rope_sin, padding_mask)
        return self.norm(x)



# ───────────────────────────── causal decoder (Mistral-style: GQA + RoPE +
# SwiGLU + RMSNorm). Copied from train_phase1_local.ipynb so the Phase 1
# decoder state_dict loads into the code that produced it.

def apply_rope_off(x, cos, sin, offset=0):
    T = x.size(2)
    c = cos[offset:offset+T].unsqueeze(0).unsqueeze(0)
    s = sin[offset:offset+T].unsqueeze(0).unsqueeze(0)
    hd = x.size(-1) // 2
    x1, x2 = x[..., :hd], x[..., hd:]
    rot = torch.cat([-x2, x1], dim=-1)
    c_e = torch.cat([c.expand(-1,-1,-1,hd), c.expand(-1,-1,-1,hd)], -1)
    s_e = torch.cat([s.expand(-1,-1,-1,hd), s.expand(-1,-1,-1,hd)], -1)
    return x * c_e + rot * s_e


class GQA(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.nqh=cfg.n_query_heads; self.nkh=cfg.n_kv_heads; self.ng=cfg.n_groups
        self.hd=cfg.head_dim; self.dm=cfg.d_model
        self.q = nn.Linear(cfg.d_model, cfg.n_query_heads * cfg.head_dim, bias=False)
        self.k = nn.Linear(cfg.d_model, cfg.n_kv_heads   * cfg.head_dim, bias=False)
        self.v = nn.Linear(cfg.d_model, cfg.n_kv_heads   * cfg.head_dim, bias=False)
        self.o = nn.Linear(cfg.n_query_heads * cfg.head_dim, cfg.d_model, bias=False)
        self.attn_drop  = nn.Dropout(cfg.dropout)
        self.resid_drop = nn.Dropout(cfg.dropout)
        self.scale = 1.0 / math.sqrt(cfg.head_dim)

    def forward(self, x, cos, sin, mask=None, kv_cache=None, offset=0):
        B, T, _ = x.shape
        q = self.q(x).view(B, T, self.nqh, self.hd).transpose(1,2)
        k = self.k(x).view(B, T, self.nkh, self.hd).transpose(1,2)
        v = self.v(x).view(B, T, self.nkh, self.hd).transpose(1,2)
        q = apply_rope_off(q, cos, sin, offset)
        k = apply_rope_off(k, cos, sin, offset)
        new_cache = (k, v)
        if self.ng > 1:
            k = k.unsqueeze(2).expand(B, self.nkh, self.ng, k.size(2), self.hd).reshape(B, self.nqh, -1, self.hd)
            v = v.unsqueeze(2).expand(B, self.nkh, self.ng, v.size(2), self.hd).reshape(B, self.nqh, -1, self.hd)
        w = torch.matmul(q, k.transpose(-2,-1)) * self.scale
        if mask is not None: w = w + mask
        w = F.softmax(w, dim=-1, dtype=torch.float32).to(q.dtype)
        w = self.attn_drop(w)
        out = torch.matmul(w, v).transpose(1,2).contiguous().view(B, T, self.dm)
        return self.resid_drop(self.o(out)), new_cache


class DecoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model); self.attn = GQA(cfg)
        self.norm2 = RMSNorm(cfg.d_model); self.ffn  = SwiGLU(cfg)
    def forward(self, x, cos, sin, mask=None, kv_cache=None, offset=0):
        h, c = self.attn(self.norm1(x), cos, sin, mask, kv_cache, offset)
        x = x + h
        return x + self.ffn(self.norm2(x)), c


class LegalDecoder(nn.Module):
    """Phase 1 causal decoder, used here as a feature extractor."""
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        self.layers = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg.n_layers)])
        self.norm = RMSNorm(cfg.d_model)
        cos, sin = precompute_rope(cfg.head_dim, cfg.rope_table_len, cfg.rope_theta)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        self.apply(LegalEncoder._init)

    def _causal_mask(self, T, device):
        cols = torch.arange(T, device=device)[None, :]
        rows = torch.arange(T, device=device)[:, None]
        m = torch.zeros((T, T), device=device).masked_fill(cols > rows, float("-inf"))
        return m.unsqueeze(0).unsqueeze(0)

    def forward(self, ids, padding_mask=None):
        T = ids.size(1)
        if T > self.rope_cos.size(0):
            raise ValueError(
                f"sequence length {T} exceeds the RoPE table "
                f"({self.rope_cos.size(0)}).")
        x = self.drop(self.tok_emb(ids))
        mask = self._causal_mask(T, ids.device)
        for layer in self.layers:
            x, _ = layer(x, self.rope_cos, self.rope_sin, mask, None, 0)
        return self.norm(x)


def decoder_cfg_from_sd(sd, n_query_heads=12, max_seq_len=512, dropout=0.1,
                        rope_theta=10000.0, rope_table_len=None):
    c = BackboneCfg()
    c.vocab_size, c.d_model = sd["tok_emb.weight"].shape
    c.n_layers = max(int(m.group(1)) for k in sd
                     if (m := re.match(r"layers\.(\d+)\.", k))) + 1
    c.d_ff = sd["layers.0.ffn.gate.weight"].shape[0]
    c.n_query_heads = n_query_heads
    c.head_dim = c.d_model // n_query_heads
    c.n_kv_heads = sd["layers.0.attn.k.weight"].shape[0] // c.head_dim
    if c.n_query_heads % c.n_kv_heads:
        raise ValueError(f"n_query_heads {c.n_query_heads} not divisible by "
                         f"inferred n_kv_heads {c.n_kv_heads}")
    c.n_groups = c.n_query_heads // c.n_kv_heads
    c.n_heads = c.n_query_heads
    c.max_seq_len, c.dropout, c.rope_theta = max_seq_len, dropout, rope_theta
    c.rope_table_len = rope_table_len or max(max_seq_len, 2 * max_seq_len)
    return c


class CJPEDecoderClassifier(nn.Module):
    """Causal backbone + pooling + linear head.

    Causal attention means position i never attends past i, so trailing PAD
    cannot affect any real token — which is why mean-pooling over real
    positions is safe even though the decoder has no padding-mask argument.
    `last` takes the final real token, the only position that has seen the
    whole window, and is the conventional readout for a causal model."""
    def __init__(self, backbone, d_model, n_classes=2, p_drop=0.1, pool="last"):
        super().__init__()
        self.backbone = backbone
        self.pool = pool
        self.drop = nn.Dropout(p_drop)
        self.head = nn.Linear(d_model, n_classes)
        nn.init.normal_(self.head.weight, std=0.02)
        nn.init.zeros_(self.head.bias)
        self._lens = None            # set by the training loop each batch

    def pooled(self, ids, pad_mask):
        h = self.backbone(ids)
        if self.pool == "last":
            lens = (~pad_mask).sum(1).clamp(min=1)
            return h[torch.arange(h.size(0), device=h.device), lens - 1]
        keep = (~pad_mask).unsqueeze(-1).to(h.dtype)
        return (h * keep).sum(1) / keep.sum(1).clamp(min=1)

    def forward(self, ids, pad_mask):
        return self.head(self.drop(self.pooled(ids, pad_mask)))


def load_phase1_decoder(cfg: CJPEConfig, n_layers_keep=None, pool="last",
                        n_query_heads=12, verbose=True):
    ck = torch.load(cfg.dec_ckpt, map_location="cpu", weights_only=False)
    sd = {(k[7:] if k.startswith("module.") else k): v
          for k, v in (ck["model"] if "model" in ck else ck).items()}
    # ---- is this file really the GQA causal decoder? ----
    if "mlm_dense.weight" in sd or "mlm_head.weight" in sd:
        raise ValueError(
            f"{Path(cfg.dec_ckpt).name} has an MLM head — that is the MHA "
            f"ENCODER checkpoint. The decoder file is best.pt — or set "
            f"BACKBONE='encoder' to use this one.")
    bc = decoder_cfg_from_sd(sd, n_query_heads=n_query_heads,
                             max_seq_len=cfg.max_len, dropout=cfg.dropout,
                             rope_theta=cfg.rope_theta,
                             rope_table_len=cfg.rope_table_len)

    keep_idx = list(range(bc.n_layers))
    if n_layers_keep is not None and n_layers_keep < bc.n_layers:
        step = bc.n_layers / n_layers_keep
        keep_idx = [min(bc.n_layers - 1, int(round(i * step)))
                    for i in range(n_layers_keep)]
        bc.n_layers = n_layers_keep

    bb = LegalDecoder(bc)
    want, new = bb.state_dict(), {}
    for k in want:
        if k.startswith("layers."):
            i = int(k.split(".")[1])
            src = k.replace(f"layers.{i}.", f"layers.{keep_idx[i]}.", 1)
        else:
            src = k
        if src in sd and sd[src].shape == want[k].shape:
            new[k] = sd[src]
    missing = [k for k in want if k not in new]
    bb.load_state_dict(new, strict=False)
    if verbose:
        print(f"GQA DECODER (causal, next-token-pretrained): {bc.d_model}d x "
              f"{bc.n_layers}L, {bc.n_query_heads} query heads sharing "
              f"{bc.n_kv_heads} KV heads, d_ff={bc.d_ff}, vocab={bc.vocab_size:,}")
        print(f"  loaded {len(new)}/{len(want)} tensors from "
              f"{Path(cfg.dec_ckpt).name} (step {ck.get('step','?')}, "
              f"val_loss {ck.get('val_loss', float('nan')):.4f})")
        print(f"  context {cfg.max_len} | RoPE table {bc.rope_table_len} | "
              f"pooling '{pool}'")
        if cfg.max_len > 512:
            print(f"  NOTE: if this checkpoint was pretrained at 512, positions "
                  f"512-{cfg.max_len-1} were never seen in training.")
        if missing:
            print(f"  NOT loaded (fresh init): {missing}")
        std = bb.tok_emb.weight.std().item()
        print(f"  tok_emb std = {std:.4f} (fresh init ~0.0200)")
        if abs(std - 0.02) < 0.002:
            print("  !! near init — check the checkpoint really loaded")
    # keep only metadata: the Phase 1 checkpoint also carries its AdamW state
    # (~1 GB for 130M params) which would otherwise sit in RAM all run
    ck = {k: v for k, v in ck.items() if k not in ("model", "opt", "scaler")}
    del sd
    return CJPEDecoderClassifier(bb, bc.d_model, 2, cfg.dropout, pool), bc, ck


def backbone_cfg_from_sd(sd, n_heads=12, max_seq_len=512, dropout=0.1,
                         rope_theta=10000.0):
    c = BackboneCfg()
    c.vocab_size, c.d_model = sd["tok_emb.weight"].shape
    c.n_layers = max(int(m.group(1)) for k in sd
                     if (m := re.match(r"layers\.(\d+)\.", k))) + 1
    c.d_ff = sd["layers.0.ffn.gate.weight"].shape[0]
    c.n_heads, c.max_seq_len, c.dropout = n_heads, max_seq_len, dropout
    c.rope_theta = rope_theta
    if c.d_model % c.n_heads:
        raise ValueError(f"d_model {c.d_model} not divisible by n_heads {c.n_heads}")
    c.head_dim = c.d_model // c.n_heads
    return c


class CJPEClassifier(nn.Module):
    """Phase 1 encoder + mean-pool + linear head. The head and pooling are the
    SAME in every experiment, so differences come from the adaptation
    technique, not from the readout (which the rules say does not count as a
    technique on its own)."""
    def __init__(self, backbone, d_model, n_classes=2, p_drop=0.1):
        super().__init__()
        self.backbone = backbone
        self.drop = nn.Dropout(p_drop)
        self.head = nn.Linear(d_model, n_classes)
        nn.init.normal_(self.head.weight, std=0.02)
        nn.init.zeros_(self.head.bias)

    def pooled(self, ids, pad_mask):
        h = self.backbone(ids, pad_mask)
        keep = (~pad_mask).unsqueeze(-1).to(h.dtype)
        return (h * keep).sum(1) / keep.sum(1).clamp(min=1)

    def forward(self, ids, pad_mask):
        return self.head(self.drop(self.pooled(ids, pad_mask)))


def load_phase1_encoder(cfg: CJPEConfig, n_layers_keep: int | None = None,
                        verbose=True):
    """Builds the classifier with Phase 1 weights. n_layers_keep < depth takes
    an evenly spaced subset of layers (used for the distillation student)."""
    ck = torch.load(cfg.enc_ckpt, map_location="cpu", weights_only=False)
    sd = {(k[7:] if k.startswith("module.") else k): v
          for k, v in (ck["model"] if "model" in ck else ck).items()}
    # ---- is this file really the MHA MLM encoder? ----
    _d = sd["tok_emb.weight"].shape[1]
    _k = sd.get("layers.0.attn.k.weight")
    if "lm_head.weight" in sd and "mlm_dense.weight" not in sd:
        raise ValueError(
            f"{Path(cfg.enc_ckpt).name} is a causal DECODER checkpoint (it has "
            f"lm_head, no MLM head). The encoder file is best_enc.pt — or set "
            f"BACKBONE='decoder' to use this one.")
    if _k is not None and _k.shape[0] != _d:
        raise ValueError(
            f"{Path(cfg.enc_ckpt).name} has grouped KV heads "
            f"(k is {tuple(_k.shape)}, not {_d}x{_d}) — that is the GQA decoder, "
            f"not the MHA encoder.")
    bc = backbone_cfg_from_sd(sd, n_heads=cfg.n_heads, max_seq_len=cfg.max_len,
                              dropout=cfg.dropout, rope_theta=cfg.rope_theta)

    keep_idx = list(range(bc.n_layers))
    if n_layers_keep is not None and n_layers_keep < bc.n_layers:
        step = bc.n_layers / n_layers_keep
        keep_idx = [min(bc.n_layers - 1, int(round(i * step)))
                    for i in range(n_layers_keep)]
        bc.n_layers = n_layers_keep

    bb = LegalEncoder(bc)
    want = bb.state_dict()
    new = {}
    for k in want:
        if k.startswith("layers."):
            i = int(k.split(".")[1])
            src = k.replace(f"layers.{i}.", f"layers.{keep_idx[i]}.", 1)
        else:
            src = k
        if src in sd and sd[src].shape == want[k].shape:
            new[k] = sd[src]
    missing = [k for k in want if k not in new]
    bb.load_state_dict(new, strict=False)
    if verbose:
        print(f"MHA ENCODER (bidirectional, MLM-pretrained): {bc.d_model}d x "
              f"{bc.n_layers}L x {bc.n_heads} heads, d_ff={bc.d_ff}, "
              f"vocab={bc.vocab_size:,}")
        print(f"  loaded {len(new)}/{len(want)} tensors from "
              f"{Path(cfg.enc_ckpt).name} (step {ck.get('step','?')}, "
              f"val_loss {ck.get('val_loss', float('nan')):.4f})")
        if keep_idx != list(range(len(keep_idx))):
            print(f"  layer subset: {keep_idx}")
        if missing:
            print(f"  NOT loaded (fresh init): {missing}")
        std = bb.tok_emb.weight.std().item()
        print(f"  tok_emb std = {std:.4f} (fresh init ~0.0200)")
        if abs(std - 0.02) < 0.002:
            print("  !! near init — check that the checkpoint really loaded")
    model = CJPEClassifier(bb, bc.d_model, 2, cfg.dropout)
    # keep only metadata: the Phase 1 checkpoint also carries its AdamW state
    # (~1 GB for 130M params) which would otherwise sit in RAM all run
    ck = {k: v for k, v in ck.items() if k not in ("model", "opt", "scaler")}
    del sd
    return model, bc, ck

def load_phase1_model(cfg: CJPEConfig, n_layers_keep=None, verbose=True):
    """Builds the classifier on whichever Phase 1 model cfg.backbone names.

    Both paths expose the same surface — .backbone.layers, .backbone.norm,
    .head, .pooled(ids, pad_mask), forward(ids, pad_mask) — so every
    adaptation technique works unchanged on either."""
    if cfg.backbone == "decoder":
        return load_phase1_decoder(cfg, n_layers_keep=n_layers_keep,
                                   pool=cfg.dec_pool,
                                   n_query_heads=cfg.n_query_heads,
                                   verbose=verbose)
    if cfg.backbone == "encoder":
        return load_phase1_encoder(cfg, n_layers_keep=n_layers_keep,
                                   verbose=verbose)
    raise ValueError(f"cfg.backbone must be 'encoder' or 'decoder', "
                     f"got {cfg.backbone!r}")

### Library 4/6 — adaptation modules (LoRA / AdaLoRA / adapters)

In [ ]:
def param_report(model, label=""):
    tot = sum(p.numel() for p in model.parameters())
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"{label}trainable {tr:,} / {tot:,} = {100*tr/max(tot,1):.3f}%")
    return tr, tot


def freeze_all(model):
    for p in model.parameters():
        p.requires_grad = False


def unfreeze_head(model):
    for p in model.head.parameters():
        p.requires_grad = True


class LoRALinear(nn.Module):
    """y = W0 x + (alpha/r) * B(A(x)).  W0 frozen; A, B trainable."""
    def __init__(self, base: nn.Linear, r=8, alpha=16, p_drop=0.0):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False
        self.r, self.alpha = r, alpha
        self.scaling = alpha / r
        self.lora_A = nn.Parameter(torch.zeros(r, base.in_features))
        self.lora_B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)        # ΔW = 0 at init
        self.drop = nn.Dropout(p_drop)

    def forward(self, x):
        out = self.base(x)
        return out + self.drop(x) @ self.lora_A.T @ self.lora_B.T * self.scaling


class AdaLoRALinear(nn.Module):
    """SVD-style ΔW = P diag(Λ) Q with a prunable rank budget.

    Implements the parts of AdaLoRA that carry the idea: the triplet
    parameterisation, a sensitivity-based importance score per rank with
    exponential smoothing, a cubic budget schedule that prunes rank over
    training, and an orthogonality penalty on P and Q. It does NOT implement
    global budget redistribution across modules — each module anneals to the
    same target rank. Say so in the report rather than claiming full AdaLoRA.
    """
    def __init__(self, base: nn.Linear, r_init=12, alpha=16, p_drop=0.0):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False
        self.r_init, self.alpha = r_init, alpha
        self.scaling = alpha / r_init
        self.P = nn.Parameter(torch.zeros(base.out_features, r_init))
        self.E = nn.Parameter(torch.zeros(r_init))
        self.Q = nn.Parameter(torch.zeros(r_init, base.in_features))
        nn.init.normal_(self.P, std=0.02)
        nn.init.normal_(self.Q, std=0.02)
        nn.init.zeros_(self.E)             # ΔW = 0 at init
        self.register_buffer("rank_mask", torch.ones(r_init))
        self.register_buffer("ipt", torch.zeros(r_init))
        self.drop = nn.Dropout(p_drop)

    def forward(self, x):
        e = self.E * self.rank_mask
        return self.base(x) + (self.drop(x) @ self.Q.T * e) @ self.P.T * self.scaling

    @torch.no_grad()
    def update_importance(self, beta=0.85):
        """Sensitivity |λ * dL/dλ|, exponentially smoothed. Call after backward."""
        if self.E.grad is None:
            return
        s = (self.E * self.E.grad).abs()
        self.ipt.mul_(beta).add_(s, alpha=1 - beta)

    @torch.no_grad()
    def set_rank(self, k):
        """Keep the k most important directions."""
        k = max(1, min(int(k), self.r_init))
        keep = torch.topk(self.ipt, k).indices
        m = torch.zeros_like(self.rank_mask)
        m[keep] = 1.0
        self.rank_mask.copy_(m)

    def ortho_penalty(self):
        I_p = torch.eye(self.P.size(1), device=self.P.device)
        I_q = torch.eye(self.Q.size(0), device=self.Q.device)
        return ((self.P.T @ self.P - I_p).pow(2).mean()
                + (self.Q @ self.Q.T - I_q).pow(2).mean())


class Adapter(nn.Module):
    """Houlsby bottleneck: x + Up(GELU(Down(LN(x)))), up zero-init."""
    def __init__(self, d_model, bottleneck=64):
        super().__init__()
        self.norm = RMSNorm(d_model)
        self.down = nn.Linear(d_model, bottleneck)
        self.up = nn.Linear(bottleneck, d_model)
        nn.init.normal_(self.down.weight, std=0.02); nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self, x):
        return x + self.up(F.gelu(self.down(self.norm(x))))


class AdaptedBlock(nn.Module):
    """Wraps a frozen EncoderBlock, inserting an adapter after each sublayer."""
    def __init__(self, block: EncoderBlock, d_model, bottleneck=64):
        super().__init__()
        self.block = block
        self.adapter_attn = Adapter(d_model, bottleneck)
        self.adapter_ffn = Adapter(d_model, bottleneck)
    def forward(self, x, cos, sin, padding_mask=None):
        b = self.block
        x = x + self.adapter_attn(b.attn(b.norm1(x), cos, sin, padding_mask))
        return x + self.adapter_ffn(b.ffn(b.norm2(x)))


class AdaptedDecoderBlock(nn.Module):
    """Wraps a frozen decoder block, inserting an adapter after each sublayer.
    The decoder's attention returns (out, kv_cache), so this keeps that shape."""
    def __init__(self, block, d_model, bottleneck=64):
        super().__init__()
        self.block = block
        self.adapter_attn = Adapter(d_model, bottleneck)
        self.adapter_ffn = Adapter(d_model, bottleneck)
    def forward(self, x, cos, sin, mask=None, kv_cache=None, offset=0):
        b = self.block
        h, c = b.attn(b.norm1(x), cos, sin, mask, kv_cache, offset)
        x = x + self.adapter_attn(h)
        return x + self.adapter_ffn(b.ffn(b.norm2(x))), c


# ---- injection helpers ----

ATTN_TARGETS = ("q", "k", "v", "o")
FFN_TARGETS = ("gate", "up", "down")


def inject_lora(model, r=8, alpha=16, targets=ATTN_TARGETS, ffn_targets=(),
                p_drop=0.0, cls=LoRALinear, **kw):
    """Replaces the named nn.Linear modules inside each block. Returns the list
    of injected modules."""
    injected = []
    for blk in model.backbone.layers:
        for name in targets:
            base = getattr(blk.attn, name)
            new = cls(base, r, alpha, p_drop, **kw) if cls is not LoRALinear else \
                  cls(base, r, alpha, p_drop)
            setattr(blk.attn, name, new); injected.append(new)
        for name in ffn_targets:
            base = getattr(blk.ffn, name)
            new = cls(base, r, alpha, p_drop, **kw) if cls is not LoRALinear else \
                  cls(base, r, alpha, p_drop)
            setattr(blk.ffn, name, new); injected.append(new)
    return injected


def inject_adalora(model, r_init=12, alpha=16, targets=ATTN_TARGETS,
                   ffn_targets=(), p_drop=0.0):
    injected = []
    for blk in model.backbone.layers:
        for name in targets:
            new = AdaLoRALinear(getattr(blk.attn, name), r_init, alpha, p_drop)
            setattr(blk.attn, name, new); injected.append(new)
        for name in ffn_targets:
            new = AdaLoRALinear(getattr(blk.ffn, name), r_init, alpha, p_drop)
            setattr(blk.ffn, name, new); injected.append(new)
    return injected


def inject_adapters(model, bottleneck=64):
    d = model.backbone.cfg.d_model
    cls = (AdaptedDecoderBlock if isinstance(model.backbone, LegalDecoder)
           else AdaptedBlock)
    model.backbone.layers = nn.ModuleList(
        [cls(b, d, bottleneck) for b in model.backbone.layers])
    return [m for m in model.backbone.layers]


def adalora_budget(step, total, r_init, r_target, warmup_frac=0.1,
                   final_frac=0.3):
    """Cubic schedule: hold r_init, prune to r_target, then hold."""
    w, f = warmup_frac * total, (1 - final_frac) * total
    if step <= w:
        return r_init
    if step >= f:
        return r_target
    p = (step - w) / max(f - w, 1)
    return int(round(r_target + (r_init - r_target) * (1 - p) ** 3))

### Library 5/6 — training, evaluation, results, submission

In [ ]:
def macro_metrics(y_true, y_pred):
    from sklearn.metrics import (f1_score, precision_score, recall_score,
                                 accuracy_score, confusion_matrix)
    return {
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "f1_per_class": f1_score(y_true, y_pred, average=None, zero_division=0).tolist(),
        "confusion": confusion_matrix(y_true, y_pred).tolist(),
    }


@torch.no_grad()
def predict(model, loader, pad_id, return_logits=True):
    model.eval()
    P, Y = [], []
    for ids, n, y in loader:
        ids = ids.to(DEVICE, non_blocking=True)
        pm = ids == pad_id
        with torch.amp.autocast(device_type=DEVICE.type,
                                enabled=(DEVICE.type == "cuda")):
            logits = model(ids, pm)
        P.append(logits.float().cpu())
        Y.append(y)
    logits = torch.cat(P).numpy()
    y = torch.cat(Y).numpy()
    return (logits, y) if return_logits else (logits.argmax(1), y)


def evaluate(model, loader, pad_id):
    logits, y = predict(model, loader, pad_id)
    m = macro_metrics(y, logits.argmax(1))
    m["_logits"], m["_y"] = logits, y
    return m


def train_model(model, train_loader, dev_loader, pad_id, cfg: CJPEConfig,
                param_groups=None, class_weight=None, on_backward=None,
                on_step=None, extra_loss=None, label="run"):
    """One training loop for every experiment.

    on_backward(step)      called after loss.backward(), before the optimizer step
    on_step(step, total)   called after each optimizer step (rank schedules etc.)
    extra_loss()           returns a scalar added to the loss (regularisers)
    """
    model.to(DEVICE)
    if param_groups is None:
        decay, no_decay, head = [], [], []
        for n_, p in model.named_parameters():
            if not p.requires_grad:
                continue
            if n_.startswith("head."):
                head.append(p)
            elif p.ndim < 2 or "norm" in n_.lower() or n_.endswith(".bias"):
                no_decay.append(p)
            else:
                decay.append(p)
        param_groups = [
            {"params": decay, "weight_decay": cfg.weight_decay, "lr": cfg.lr},
            {"params": no_decay, "weight_decay": 0.0, "lr": cfg.lr},
            {"params": head, "weight_decay": cfg.weight_decay, "lr": cfg.head_lr},
        ]
    param_groups = [g for g in param_groups if len(g["params"])]
    opt = torch.optim.AdamW(param_groups, betas=(0.9, 0.999))
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))

    steps_per_epoch = max(1, math.ceil(len(train_loader) / cfg.grad_accum))
    total = steps_per_epoch * cfg.epochs
    warm = max(1, int(cfg.warmup_frac * total))
    base_lrs = [g["lr"] for g in opt.param_groups]

    def lr_at(s):
        if s < warm:
            return (s + 1) / warm
        p = (s - warm) / max(1, total - warm)
        return 0.5 * (1 + math.cos(math.pi * min(p, 1.0)))

    cw = None if class_weight is None else torch.tensor(
        class_weight, dtype=torch.float32, device=DEVICE)

    hist, best, best_state, no_imp, step = [], -1.0, None, 0, 0
    t0 = time.time()
    micro = 0
    model.train()
    stop = False
    for ep in range(cfg.epochs):
        if stop:
            break
        for ids, n, y in train_loader:
            ids = ids.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            pm = ids == pad_id
            with torch.amp.autocast(device_type=DEVICE.type,
                                    enabled=(DEVICE.type == "cuda")):
                logits = model(ids, pm)
                loss = F.cross_entropy(logits.float(), y, weight=cw,
                                       label_smoothing=cfg.label_smoothing)
                if extra_loss is not None:
                    loss = loss + extra_loss()
                loss = loss / cfg.grad_accum
            scaler.scale(loss).backward()
            micro += 1
            if micro % cfg.grad_accum:
                continue

            scaler.unscale_(opt)
            if on_backward is not None:
                on_backward(step)
            torch.nn.utils.clip_grad_norm_(
                [p for g in opt.param_groups for p in g["params"]],
                cfg.max_grad_norm)
            f = lr_at(step)
            for g, b in zip(opt.param_groups, base_lrs):
                g["lr"] = b * f
            scaler.step(opt); scaler.update()
            opt.zero_grad(set_to_none=True)
            step += 1
            if on_step is not None:
                on_step(step, total)

            if step % cfg.eval_every == 0 or step == total:
                m = evaluate(model, dev_loader, pad_id)
                hist.append({"step": step, "macro_f1": m["macro_f1"],
                             "accuracy": m["accuracy"]})
                flag = ""
                if m["macro_f1"] > best:
                    best = m["macro_f1"]; no_imp = 0
                    best_state = {k: v.detach().cpu().clone()
                                  for k, v in model.state_dict().items()}
                    flag = "  *best*"
                else:
                    no_imp += 1
                print(f"  [{label}] step {step}/{total} ep{ep+1} "
                      f"dev macro-F1 {m['macro_f1']:.4f} acc {m['accuracy']:.4f}"
                      f"{flag}  ({time.time()-t0:.0f}s)")
                model.train()
                if no_imp >= cfg.patience:
                    print(f"  [{label}] early stop: {cfg.patience} evals without "
                          f"improvement")
                    stop = True
                    break

    if best_state is not None:
        model.load_state_dict(best_state)
    return {"best_dev_macro_f1": best, "history": hist,
            "train_seconds": time.time() - t0, "steps": step}

def record_result(cfg: CJPEConfig, name, technique, metrics, trainable, total,
                  train_info, notes="", extra=None):
    path = Path(cfg.results_json)
    allr = json.loads(path.read_text()) if path.exists() else []
    allr = [r for r in allr if r.get("name") != name]
    row = {
        "name": name, "technique": technique,
        "macro_f1": metrics["macro_f1"], "precision": metrics["precision"],
        "recall": metrics["recall"], "accuracy": metrics["accuracy"],
        "f1_per_class": metrics["f1_per_class"], "confusion": metrics["confusion"],
        "trainable_params": int(trainable), "total_params": int(total),
        "trainable_pct": round(100 * trainable / max(total, 1), 4),
        "train_seconds": round(train_info.get("train_seconds", 0), 1),
        "steps": train_info.get("steps", 0),
        "history": train_info.get("history", []),
        "window": cfg.window, "max_len": cfg.max_len, "epochs": cfg.epochs,
        "lr": cfg.lr, "batch": cfg.batch_size * cfg.grad_accum,
        "quick": cfg.quick, "n_train": cfg.max_train, "notes": notes,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    if extra:
        row.update(extra)
    allr.append(row)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(allr, indent=1))
    print(f"recorded '{name}' -> {path}")
    return row


def save_dev_predictions(cfg: CJPEConfig, name, ids, logits, y):
    p = Path(cfg.cache_dir) / f"devpred_{name}.npz"
    np.savez(p, ids=np.array(ids), logits=logits, y=y)
    return str(p)


def load_results(cfg: CJPEConfig):
    path = Path(cfg.results_json)
    return json.loads(path.read_text()) if path.exists() else []

def best_threshold(logits, y, lo=0.05, hi=0.95, n=91):
    """Decision threshold that maximises macro-F1 on the dev logits. Phase 4
    lists calibration and post-processing as permitted, and a model whose
    predictions are skewed usually leaves several macro-F1 points here."""
    from sklearn.metrics import f1_score
    z = logits - logits.max(1, keepdims=True)
    p = np.exp(z); p = (p / p.sum(1, keepdims=True))[:, 1]
    ts = np.linspace(lo, hi, n)
    f = [f1_score(y, (p >= t).astype(int), average="macro", zero_division=0)
         for t in ts]
    i = int(np.argmax(f))
    at_half = f1_score(y, (p >= 0.5).astype(int), average="macro", zero_division=0)
    return float(ts[i]), float(f[i]), float(at_half)


def write_submission(cfg: CJPEConfig, model, tok, pad_id, out_path,
                     local_test=None, batch_size=16, threshold=0.5):
    """Writes [{"id": ..., "pred": int}, ...] for every case in test.json.

    out_path should end in prediction.json — that is the filename the
    competition expects inside the submission."""
    rows = load_split(cfg, "test", local_test)
    print(f"test cases: {len(rows)}")
    X, n = encode_split(tok, rows, pad_id, cfg, desc="test")
    loader = make_loader(X, n, None, batch_size, shuffle=False)
    logits, _ = predict(model, loader, pad_id)

    z = logits - logits.max(1, keepdims=True)
    prob = np.exp(z); prob = (prob / prob.sum(1, keepdims=True))[:, 1]
    pred = (prob >= threshold).astype(int)

    sub = [{"id": rows[i][0], "pred": int(pred[i])} for i in range(len(rows))]
    assert len(sub) == len(rows), "one prediction per test case is required"
    assert all(s["pred"] in (0, 1) for s in sub), "pred must be 0 or 1"
    assert all(isinstance(s["id"], str) for s in sub)
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(sub, indent=1))
    from collections import Counter
    mix = Counter(pred.tolist())
    print(f"wrote {out_path}")
    print(f"  {len(sub)} rows | threshold {threshold:.2f} | "
          f"predicted 0: {mix.get(0,0)}  1: {mix.get(1,0)} "
          f"({mix.get(1,0)/len(sub):.1%} positive)")
    return sub

def rebuild_from_checkpoint(path, dropout=0.0, dec_pool="last", lora_alpha=16,
                            ada_alpha=16, verbose=True):
    """Rebuilds ANY model saved by the experiment notebooks (plain, LoRA,
    AdaLoRA, adapters, distilled student; encoder or decoder) from its .pt
    file alone, by reading the module structure off the state_dict keys.

    LoRA/AdaLoRA alpha is not stored in the state_dict; the notebooks used 16.
    Load is strict, and the caller re-checks dev macro-F1 against the value
    saved at training time, so a wrong reconstruction cannot pass silently."""
    ck = torch.load(path, map_location="cpu", weights_only=False)
    sd = ck["model"]
    bd = dict(ck.get("backbone") or ck.get("cfg"))
    bc = BackboneCfg(); bc.__dict__.update(bd); bc.dropout = dropout
    is_dec = "n_kv_heads" in bd
    if is_dec:
        if getattr(bc, "rope_table_len", None) is None:
            bc.rope_table_len = 2 * bc.max_seq_len
        model = CJPEDecoderClassifier(LegalDecoder(bc), bc.d_model, 2, dropout,
                                      dec_pool)
    else:
        model = CJPEClassifier(LegalEncoder(bc), bc.d_model, 2, dropout)

    keys = list(sd)
    def _targets(suffix):
        a = tuple(n for n in ATTN_TARGETS
                  if f"backbone.layers.0.attn.{n}.{suffix}" in sd)
        f = tuple(n for n in FFN_TARGETS
                  if f"backbone.layers.0.ffn.{n}.{suffix}" in sd)
        return a, f

    kind = "plain"
    if any(k.endswith(".lora_A") for k in keys):
        r = next(sd[k] for k in keys if k.endswith(".lora_A")).shape[0]
        a, f = _targets("lora_A")
        inject_lora(model, r=r, alpha=lora_alpha, targets=a, ffn_targets=f)
        kind = f"LoRA r={r} on {a + f}"
    elif any(k.endswith(".rank_mask") for k in keys):
        r = next(sd[k] for k in keys if k.endswith(".rank_mask")).shape[0]
        a, f = _targets("P")
        inject_adalora(model, r_init=r, alpha=ada_alpha, targets=a, ffn_targets=f)
        kind = f"AdaLoRA r_init={r} on {a + f}"
    elif any(".adapter_attn." in k for k in keys):
        b = next(sd[k] for k in keys
                 if k.endswith("adapter_attn.down.weight")).shape[0]
        inject_adapters(model, bottleneck=b)
        kind = f"adapters bottleneck={b}"

    missing, unexpected = model.load_state_dict(sd, strict=False)
    missing = [m for m in missing if "rope_" not in m]
    if missing or unexpected:
        raise RuntimeError(f"{Path(path).name}: structure mismatch\n"
                           f"  missing {missing[:6]}\n  unexpected {unexpected[:6]}")
    model.eval()
    meta = {
        "name": ck.get("name", Path(path).stem),
        "technique": ck.get("technique", "?"),
        "backbone": "decoder" if is_dec else "encoder",
        "n_layers": bc.n_layers,
        "window": ck.get("window", "head_tail"),
        "max_len": ck.get("max_len", bc.max_seq_len),
        "head_frac": ck.get("head_frac", 0.5),
        "saved_dev_macro_f1": ck.get("dev_macro_f1"),
        "kind": kind,
    }
    if verbose:
        print(f"{meta['name']}: {meta['backbone']} {bc.n_layers}L, {kind}, "
              f"window={meta['window']} max_len={meta['max_len']}")
    return model, meta


def chunk_ids(tok, text, pad_id, L, max_chunks=8, max_words=30000):
    """Evenly spaced L-token chunks spanning the WHOLE judgment, start to end.
    Short documents give one chunk; long ones give up to max_chunks."""
    w = text.split()[:max_words]
    ids = tok.encode(" ".join(w), add_special_tokens=True)
    T = len(ids)
    if T <= L:
        starts = [0]
    else:
        k = min(math.ceil(T / L), max_chunks)
        starts = [0] if k == 1 else [int(round(i * (T - L) / (k - 1)))
                                     for i in range(k)]
    out = []
    for s in starts:
        c = ids[s:s + L]
        out.append(c + [pad_id] * (L - len(c)))
    return out


def encode_chunks_split(tok, rows, pad_id, L, max_chunks=8, desc=""):
    X, doc = [], []
    t0 = time.time()
    for i, (_, text, _) in enumerate(rows):
        for c in chunk_ids(tok, text, pad_id, L, max_chunks):
            X.append(c); doc.append(i)
        if desc and i and i % 500 == 0:
            print(f"  {desc} {i}/{len(rows)} docs ({time.time()-t0:.0f}s)")
    X = np.asarray(X, dtype=np.int32); doc = np.asarray(doc, dtype=np.int64)
    if desc:
        print(f"  {desc}: {len(rows)} docs -> {len(X)} chunks "
              f"(mean {len(X)/max(len(rows),1):.2f}/doc) in {time.time()-t0:.0f}s")
    return X, doc


@torch.no_grad()
def predict_array(model, X, pad_id, batch_size=32):
    model.to(DEVICE).eval()
    out = []
    for i in range(0, len(X), batch_size):
        ids = torch.from_numpy(X[i:i + batch_size].astype(np.int64)).to(DEVICE)
        with torch.amp.autocast(device_type=DEVICE.type,
                                enabled=(DEVICE.type == "cuda")):
            out.append(model(ids, ids == pad_id).float().cpu())
    return torch.cat(out).numpy()


def aggregate_chunks(chunk_logits, doc, n_docs):
    """Mean of chunk logits per document."""
    agg = np.zeros((n_docs, chunk_logits.shape[1]), dtype=np.float64)
    cnt = np.zeros(n_docs, dtype=np.float64)
    np.add.at(agg, doc, chunk_logits)
    np.add.at(cnt, doc, 1.0)
    return (agg / np.maximum(cnt, 1)[:, None]).astype(np.float32)


def window_logits(model, meta, tok, rows, pad_id, batch_size=32):
    """The single training-style window, exactly as the model was trained."""
    wcfg = CJPEConfig(window=meta["window"], max_len=meta["max_len"],
                      head_frac=meta["head_frac"])
    X, _ = encode_split(tok, rows, pad_id, wcfg)
    return predict_array(model, X, pad_id, batch_size)


def eval_views(y, views):
    """macro-F1 for each view at t=0.5 and at its dev-tuned threshold."""
    res = {}
    for k, lg in views.items():
        t, f_t, f_half = best_threshold(lg, y)
        res[k] = {"f1_at_half": f_half, "best_t": t, "f1_at_t": f_t}
    return res

### Library 6/6 — comparison and analytics

In [ ]:
import json, math
from pathlib import Path

import numpy as np


# ════════════════════════════════════════════════════ helpers

def _softmax(z):
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


def _load_pred(cfg, name):
    p = Path(cfg.cache_dir) / f"devpred_{name}.npz"
    if not p.exists():
        return None
    d = np.load(p, allow_pickle=True)
    return {"ids": d["ids"], "logits": d["logits"], "y": d["y"]}


def _macro_f1(y, p):
    from sklearn.metrics import f1_score
    return float(f1_score(y, p, average="macro", zero_division=0))


# ════════════════════════════════════════════════════ 1. did it learn?

def training_curve(info, metrics=None):
    """Dev macro-F1 against optimizer step, as a text plot."""
    h = info.get("history") or []
    if not h:
        print("no eval history recorded")
        return
    print("DID IT LEARN?  dev macro-F1 over training")
    print("-" * 58)
    vals = [e["macro_f1"] for e in h]
    lo, hi = min(vals), max(vals)
    span = max(hi - lo, 1e-6)
    for e in h:
        bar = "#" * int(1 + 34 * (e["macro_f1"] - lo) / span)
        star = "  <- best" if e["macro_f1"] == hi else ""
        print(f"  step {e['step']:>6}  {e['macro_f1']:.4f}  {bar}{star}")
    first, best = vals[0], hi
    print("-" * 58)
    print(f"  first eval {first:.4f}  ->  best {best:.4f}   "
          f"change {best - first:+.4f}")
    if best - first < 0.01:
        print("  the run barely moved from its first evaluation")
    last_third = vals[max(1, 2 * len(vals) // 3):]
    if last_third and max(last_third) < hi - 1e-9:
        print("  best was NOT in the final third — it was still improving "
              "earlier, then fell back")
    return {"first": first, "best": best}


# ════════════════════════════════════════════════════ 2. vs the floors

def vs_floors(metrics, y):
    """Is this better than predicting nothing? The references that do not
    depend on any other experiment."""
    n = len(y)
    maj = np.full(n, int(round(y.mean())))
    maj_f1 = _macro_f1(y, maj)
    rng = np.random.default_rng(0)
    coin = np.array([_macro_f1(y, rng.integers(0, 2, n)) for _ in range(40)])

    print("\nIS IT BETTER THAN NOTHING?")
    print("-" * 58)
    print(f"  this experiment        macro-F1 {metrics['macro_f1']:.4f}")
    print(f"  always-majority-class  macro-F1 {maj_f1:.4f}   "
          f"({metrics['macro_f1'] - maj_f1:+.4f})")
    print(f"  random coin flip       macro-F1 {coin.mean():.4f} "
          f"+/- {coin.std():.4f}  ({metrics['macro_f1'] - coin.mean():+.4f})")
    half = 1.96 * math.sqrt(0.25 / n)
    print(f"\n  95% CI half-width on accuracy at n={n}: +/-{half:.4f}")
    print(f"  so a macro-F1 gap under ~{half:.3f} is not distinguishable "
          f"from noise")
    return {"majority": maj_f1, "coin": float(coin.mean())}


# ════════════════════════════════════════════════════ 3. is it balanced?

def balance_report(metrics, y):
    """On a balanced dev set macro-F1 == accuracy for a balanced predictor.
    A gap means the model is favouring one class, which is usually the single
    biggest recoverable loss of macro-F1."""
    cm = np.array(metrics["confusion"])
    pred_rate = cm[:, 1].sum() / cm.sum()
    true_rate = cm[1].sum() / cm.sum()
    gap = metrics["accuracy"] - metrics["macro_f1"]

    print("\nIS IT BALANCED?")
    print("-" * 58)
    print(f"  true  positives in dev : {true_rate:.1%}")
    print(f"  predicted as positive  : {pred_rate:.1%}")
    print(f"  accuracy {metrics['accuracy']:.4f} - macro-F1 "
          f"{metrics['macro_f1']:.4f} = {gap:+.4f}")
    if gap > 0.03:
        print("  a gap this size means predictions are skewed toward one class;")
        print("  the threshold sweep below says how much that is costing you")
    else:
        print("  predictions are close to balanced")
    for c in (0, 1):
        tp = cm[c, c]; fp = cm[1 - c, c]; fn = cm[c, 1 - c]
        pr = tp / max(tp + fp, 1); rc = tp / max(tp + fn, 1)
        print(f"  class {c}: precision {pr:.4f}  recall {rc:.4f}  "
              f"support {cm[c].sum()}")


# ════════════════════════════════════════════════════ 4. threshold sweep

def threshold_sweep(logits, y, show=9):
    """Phase 4 explicitly allows calibration and post-processing. This asks
    what macro-F1 the SAME model would reach at a different decision
    threshold — a free gain that costs no retraining."""
    prob = _softmax(logits)[:, 1]
    ts = np.linspace(0.05, 0.95, 91)
    f1s = np.array([_macro_f1(y, (prob >= t).astype(int)) for t in ts])
    i = int(f1s.argmax())
    at_half = _macro_f1(y, (prob >= 0.5).astype(int))

    print("\nWOULD A DIFFERENT THRESHOLD HELP?")
    print("-" * 58)
    step = max(1, len(ts) // show)
    for j in range(0, len(ts), step):
        mark = "  <- best" if j == i else ("  <- argmax (default)"
                                           if abs(ts[j] - 0.5) < 1e-9 else "")
        bar = "#" * int(40 * f1s[j])
        print(f"  t={ts[j]:.2f}  macro-F1 {f1s[j]:.4f}  {bar}{mark}")
    print("-" * 58)
    print(f"  default t=0.50 : {at_half:.4f}")
    print(f"  best    t={ts[i]:.2f} : {f1s[i]:.4f}   ({f1s[i] - at_half:+.4f})")
    if f1s[i] - at_half > 0.01:
        print("  worth reporting: tuning the threshold on dev is legitimate")
        print("  post-processing under Phase 4's rules, and costs nothing.")
        print("  Fit it on dev, then apply the SAME value to test.")
    return {"best_t": float(ts[i]), "f1_at_best_t": float(f1s[i]),
            "f1_at_half": float(at_half)}


# ════════════════════════════════════════════════════ 5. where does it fail?

def error_profile(logits, y, ids=None, dev_rows=None):
    pred = logits.argmax(1)
    prob = _softmax(logits)[:, 1]
    cm = np.zeros((2, 2), dtype=int)
    for t, p in zip(y, pred):
        cm[t, p] += 1

    print("\nWHERE DOES IT FAIL?")
    print("-" * 58)
    print("  confusion (rows = true, cols = predicted)")
    print(f"              pred 0   pred 1")
    print(f"    true 0  {cm[0,0]:>8} {cm[0,1]:>8}")
    print(f"    true 1  {cm[1,0]:>8} {cm[1,1]:>8}")
    fp, fn = int(cm[0, 1]), int(cm[1, 0])
    print(f"  false positives {fp}   false negatives {fn}")
    if fp + fn:
        print(f"  {100*fp/(fp+fn):.0f}% of errors are false positives")

    conf = np.maximum(prob, 1 - prob)
    wrong = pred != y
    if wrong.any() and (~wrong).any():
        print(f"\n  mean confidence: correct {conf[~wrong].mean():.3f}   "
              f"wrong {conf[wrong].mean():.3f}")
        if conf[wrong].mean() >= conf[~wrong].mean() - 0.01:
            print("  the model is as confident when wrong as when right —")
            print("  its probabilities carry little information")

    # calibration
    edges = np.linspace(0, 1, 11)
    ece = 0.0
    rows = []
    for i in range(10):
        m = (prob >= edges[i]) & (prob < edges[i + 1] + (i == 9) * 1e-9)
        if m.sum() == 0:
            continue
        c, a = prob[m].mean(), (y[m] == 1).mean()
        ece += m.sum() / len(y) * abs(c - a)
        rows.append((edges[i], edges[i + 1], int(m.sum()), c, a))
    print(f"\n  calibration (expected calibration error {ece:.4f})")
    for lo, hi, n_, c, a in rows:
        print(f"    [{lo:.1f},{hi:.1f})  n={n_:>5}  predicted {c:.3f}  "
              f"observed {a:.3f}")

    # length buckets — the long-document question
    if dev_rows is not None and ids is not None:
        idmap = {str(r[0]): r[1] for r in dev_rows}
        wl = np.array([len(idmap.get(str(i), "").split()) for i in ids])
        if (wl > 0).any():
            print("\n  accuracy by document length "
                  "(does the 512-token window break down?)")
            qs = sorted(set(np.quantile(wl[wl > 0], [0, .25, .5, .75, 1.0]).tolist()))
            if len(qs) < 2:
                print(f"    all dev docs are {int(qs[0])} words")
            else:
                accs = []
                for i in range(len(qs) - 1):
                    lo, hi = qs[i], qs[i + 1]
                    last = i == len(qs) - 2
                    m = (wl >= lo) & ((wl <= hi) if last else (wl < hi)) & (wl > 0)
                    if m.sum() == 0:
                        continue
                    a = (pred[m] == y[m]).mean()
                    accs.append(a)
                    print(f"    {int(lo):>6}-{int(hi):>6} words  n={m.sum():>5}  "
                          f"acc {a:.4f}")
                if len(accs) >= 2 and accs[0] - accs[-1] > 0.03:
                    print("    accuracy falls on the longest documents — the")
                    print("    window is missing decisive material in those cases")
    return {"ece": float(ece), "confusion": cm.tolist()}


# ════════════════════════════════════════════════════ one call

def experiment_report(cfg, name, info, metrics, dev_rows=None):
    """Everything about THIS experiment, and nothing about the others."""
    y = metrics["_y"]
    logits = metrics["_logits"]
    d = _load_pred(cfg, name)
    ids = d["ids"] if d is not None else None

    print("=" * 58)
    print(f"EXPERIMENT REPORT — {name}")
    print("=" * 58)
    print(f"  dev macro-F1 {metrics['macro_f1']:.4f}   "
          f"precision {metrics['precision']:.4f}   "
          f"recall {metrics['recall']:.4f}   "
          f"accuracy {metrics['accuracy']:.4f}")
    print(f"  per-class F1 {[round(x,4) for x in metrics['f1_per_class']]}")
    print(f"  trained {info.get('steps','?')} steps in "
          f"{info.get('train_seconds',0)/60:.1f} min")
    print()
    training_curve(info, metrics)
    vs_floors(metrics, y)
    balance_report(metrics, y)
    thr = threshold_sweep(logits, y)
    error_profile(logits, y, ids, dev_rows)
    print("\n" + "=" * 58)
    print("What this experiment demonstrated, and whether the technique was")
    print("worth it, is yours to write.")
    print("=" * 58)
    return thr


# ════════════════════════════════════════════════════ opt-in: all six

def compare_all(cfg):
    """Only for the final write-up, once several experiments are recorded."""
    path = Path(cfg.results_json)
    rows = json.loads(path.read_text()) if path.exists() else []
    if len(rows) < 2:
        print(f"{len(rows)} experiment(s) recorded — nothing to compare yet.")
        return rows
    rows = sorted(rows, key=lambda r: -r["macro_f1"])
    w = max(len(r["name"]) for r in rows) + 2
    print("=" * (w + 54))
    print("ALL EXPERIMENTS RECORDED SO FAR (dev)")
    print("=" * (w + 54))
    print(f"{'experiment':<{w}}{'macro-F1':>10}{'prec':>7}{'rec':>7}"
          f"{'train %':>9}{'mins':>7}")
    print("-" * (w + 54))
    for r in rows:
        print(f"{r['name']:<{w}}{r['macro_f1']:>10.4f}{r['precision']:>7.4f}"
              f"{r['recall']:>7.4f}{r['trainable_pct']:>9.3f}"
              f"{r['train_seconds']/60:>7.1f}")
    print("-" * (w + 54))
    base = next((r for r in rows if r["name"] == "01_full_finetune"), None)
    if base:
        print(f"\nretained vs full fine-tuning ({base['macro_f1']:.4f}):")
        for r in rows:
            if r["name"] == base["name"]:
                continue
            print(f"  {r['name']:<24}{100*r['macro_f1']/max(base['macro_f1'],1e-9):>7.1f}%"
                  f"   at {r['trainable_params']/max(base['trainable_params'],1):.4f}x "
                  f"the trainable parameters")
    return rows


def mcnemar(cfg, name_a, name_b):
    """Paired test on the same dev cases — use when writing the comparison."""
    pa, pb = _load_pred(cfg, name_a), _load_pred(cfg, name_b)
    if pa is None or pb is None:
        print("missing saved dev predictions for one of these runs")
        return
    assert np.array_equal(pa["y"], pb["y"])
    y = pa["y"]
    a_ok = pa["logits"].argmax(1) == y
    b_ok = pb["logits"].argmax(1) == y
    n01 = int((~a_ok & b_ok).sum()); n10 = int((a_ok & ~b_ok).sum())
    if n01 + n10 == 0:
        print("identical predictions")
        return
    chi2 = (abs(n10 - n01) - 1) ** 2 / (n01 + n10)
    p = math.erfc(math.sqrt(chi2 / 2))
    print(f"McNemar {name_a} vs {name_b}: only-A {n10}, only-B {n01}, "
          f"chi2 {chi2:.3f}, p {p:.4f} "
          f"({'significant' if p < 0.05 else 'not significant'})")
    return p

## 3 — Configuration

In [ ]:
# everything this phase produces lands in one folder
OUT_DIR = f"{WORK}/cs781_phase4"
os.makedirs(OUT_DIR, exist_ok=True)
print("outputs ->", OUT_DIR)

cfg = CJPEConfig(
    drive_dir      = DRIVE,
    enc_ckpt       = ENC_PATH or "",
    tokenizer_json = TOK_JSON_PATH,
    cache_dir      = f"{OUT_DIR}/cache",
    results_json   = f"{OUT_DIR}/results.json",
    hf_token       = HF_TOKEN,

    # ---- held CONSTANT across all six experiments ----
    backbone   = BACKBONE,      # "decoder" (Phase 1 GQA LM) or "encoder" (MLM)
    dec_ckpt   = DEC_PATH or "",
    dec_pool   = "last",        # causal readout: "last" or "mean"
    n_query_heads  = 12,
    rope_table_len = ROPE_TABLE_LEN,
    window     = "head_tail",   # see the ablation in notebook 01
    max_len    = MAX_LEN,
    head_frac  = 0.5,

    # ---- set quick=True for a ~10 min smoke run first ----
    quick      = False,
    max_train  = None,          # None = all 37,387 training cases
    epochs     = 2,
    batch_size = 8,
    grad_accum = 2,
)

cfg.lr      = 5e-5
cfg.head_lr = 1e-3
STUDENT_LAYERS = 9      # teacher: 19L decoder / 14L encoder
KD_TEMP      = 3.0      # softmax temperature on both sides
KD_ALPHA     = 0.7      # weight on the KL term; (1-alpha) on hard-label CE

set_seed(cfg.seed)
tokenizer, PAD_ID, BASE_VOCAB = load_tokenizer(cfg)
print(f"tokenizer vocab {BASE_VOCAB:,} | PAD_ID {PAD_ID} | "
      f"decode -> {tokenizer.decode([PAD_ID])!r} (empty is correct)")

outputs -> /content/drive/MyDrive/cs781_phase4
tokenizer vocab 40,000 | PAD_ID 0 | decode -> '' (empty is correct)


## 4 — Data

Tokenised once and cached to Drive; every other notebook reuses the same arrays.

In [ ]:
data = prepare_data(cfg, tokenizer, PAD_ID, ("train", "dev"))
Xtr, ntr, ytr, idtr = data["train"]
Xde, nde, yde, idde = data["dev"]

print(f"train {Xtr.shape[0]:,} docs (pos rate {ytr.mean():.1%})")
print(f"dev   {Xde.shape[0]:,} docs (pos rate {yde.mean():.1%})")

train_loader = make_loader(Xtr, ntr, ytr, cfg.batch_size, True, cfg.seed)
dev_loader   = make_loader(Xde, nde, yde, cfg.batch_size * 2, False)

counts = np.bincount(ytr, minlength=2)
CLASS_W = (counts.sum() / (2 * np.maximum(counts, 1))).tolist()
print(f"train class counts {counts.tolist()} -> CE weights "
      f"[{CLASS_W[0]:.3f}, {CLASS_W[1]:.3f}]")

[cache hit ] train: 37387 docs  (X_train_head_tail_512_0.50.npy)
[cache hit ] dev: 1988 docs  (X_dev_head_tail_512_0.50.npy)
train 37,387 docs (pos rate 41.0%)
dev   1,988 docs (pos rate 50.0%)
train class counts [22067, 15320] -> CE weights [0.847, 1.220]


## 5 — Knowledge distillation (mandatory)

Two things to keep straight for the viva. The KL term is multiplied by `T^2` because softening by `T` scales the gradients by `1/T^2` — without it the distillation signal shrinks as you raise the temperature. And the student must be *smaller*, which the phase requires: the cell asserts the parameter counts rather than taking it on trust.

The interesting comparison is not whether the student beats the teacher — it usually will not — but how much of the teacher's macro-F1 it keeps at roughly half the depth.

In [ ]:
import torch.nn.functional as TF

TEACHER_CKPT = f"{OUT_DIR}/cjpe_teacher_full_ft.pt"
assert os.path.exists(TEACHER_CKPT), (
    "Run notebook 01 (full fine-tuning) first — it saves the teacher.")

teacher, tcfg, _ = load_phase1_model(cfg, verbose=False)
teacher.load_state_dict(torch.load(TEACHER_CKPT, map_location="cpu")["model"])
teacher.to(DEVICE).eval()
for p in teacher.parameters():
    p.requires_grad = False

depth = tcfg.n_layers
keep = min(STUDENT_LAYERS, depth - 1)        # the student must be shallower
print(f"teacher depth {depth} -> student depth {keep} "
      f"({100*keep/depth:.0f}% of the layers)")
if keep != STUDENT_LAYERS:
    print(f"STUDENT_LAYERS={STUDENT_LAYERS} is not smaller than the teacher's "
          f"{depth} layers — using {keep} instead.")
student, scfg, _ = load_phase1_model(cfg, n_layers_keep=keep)

n_t = sum(p.numel() for p in teacher.parameters())
n_s = sum(p.numel() for p in student.parameters())
print(f"\nteacher {tcfg.n_layers}L {n_t/1e6:.1f}M  ->  "
      f"student {scfg.n_layers}L {n_s/1e6:.1f}M  ({100*n_s/n_t:.1f}% of teacher)")
assert n_s < n_t, "the student must be smaller than the teacher"

t_dev = evaluate(teacher, dev_loader, PAD_ID)
print(f"teacher dev macro-F1: {t_dev['macro_f1']:.4f}")


class DistillWrapper(torch.nn.Module):
    """Student + frozen teacher. Exposes the student's logits so the shared
    training loop is unchanged; the KL term is added through extra_loss."""
    def __init__(self, student, teacher, T, alpha):
        super().__init__()
        self.student, self.teacher = student, teacher
        self.T, self.alpha = T, alpha
        self._kd = torch.tensor(0.0)

    def forward(self, ids, pm):
        s = self.student(ids, pm)
        with torch.no_grad():
            t = self.teacher(ids, pm)
        T = self.T
        self._kd = TF.kl_div(TF.log_softmax(s.float() / T, -1),
                             TF.softmax(t.float() / T, -1),
                             reduction="batchmean") * (T * T)
        return s

    @property
    def head(self):
        return self.student.head


model = DistillWrapper(student, teacher, KD_TEMP, KD_ALPHA).to(DEVICE)
print(f"student trainable "
      f"{sum(p.numel() for p in model.student.parameters() if p.requires_grad):,}")

# total loss = (1-alpha)*CE(hard) + alpha*T^2*KL(student||teacher)
set_seed(cfg.seed)
info = train_model(
    model, train_loader, dev_loader, PAD_ID, cfg,
    class_weight=[w * (1 - KD_ALPHA) for w in CLASS_W],
    extra_loss=lambda: KD_ALPHA * model._kd,
    label="06_distill")

print(f"\nteacher {t_dev['macro_f1']:.4f}  ->  student "
      f"{info['best_dev_macro_f1']:.4f}  "
      f"(retained {100*info['best_dev_macro_f1']/max(t_dev['macro_f1'],1e-9):.1f}% "
      f"at {100*n_s/n_t:.0f}% of the parameters)")

teacher depth 14 -> student depth 9 (64% of the layers)
MHA ENCODER (bidirectional, MLM-pretrained): 768d x 9L x 12 heads, d_ff=2048, vocab=40,001
  loaded 83/83 tensors from best_enc.pt (step 29000, val_loss 1.6970)
  layer subset: [0, 2, 3, 5, 6, 8, 9, 11, 12]
  tok_emb std = 0.0267 (fresh init ~0.0200)

teacher 14L 129.8M  ->  student 9L 94.4M  (72.7% of teacher)
teacher dev macro-F1: 0.6908
student trainable 94,437,890
  [06_distill] step 400/4674 ep1 dev macro-F1 0.5523 acc 0.6016  *best*  (264s)
  [06_distill] step 800/4674 ep1 dev macro-F1 0.6017 acc 0.6066  *best*  (530s)
  [06_distill] step 1200/4674 ep1 dev macro-F1 0.6093 acc 0.6197  *best*  (796s)
  [06_distill] step 1600/4674 ep1 dev macro-F1 0.6563 acc 0.6630  *best*  (1062s)
  [06_distill] step 2000/4674 ep1 dev macro-F1 0.6801 acc 0.6811  *best*  (1328s)
  [06_distill] step 2400/4674 ep2 dev macro-F1 0.6719 acc 0.6720  (1593s)
  [06_distill] step 2800/4674 ep2 dev macro-F1 0.6808 acc 0.6811  *best*  (1860s)
  [06_distil

## 6 — Evaluate and record

In [ ]:
metrics = evaluate(model, dev_loader, PAD_ID)
trainable = sum(p.numel() for p in model.student.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.student.parameters())
print(f"student: trainable {trainable:,} / {total:,}")

print(f"\ndev macro-F1 {metrics['macro_f1']:.4f} | "
      f"precision {metrics['precision']:.4f} | recall {metrics['recall']:.4f} | "
      f"accuracy {metrics['accuracy']:.4f}")
print("per-class F1:", [round(x, 4) for x in metrics["f1_per_class"]])

record_result(cfg, "06_distillation", "Knowledge distillation", metrics,
              trainable, total, info,
              notes="teacher=01 full FT, student=%dL subset" % scfg.n_layers,
              extra={"teacher_macro_f1": t_dev["macro_f1"]})
save_dev_predictions(cfg, "06_distillation", idde,
                     metrics["_logits"], metrics["_y"])

# ---- save the STUDENT weights (the teacher is already saved by notebook 01) ----
wpath = f"{OUT_DIR}/06_distillation.pt"
torch.save({
    "model":            model.student.state_dict(),
    "name":             "06_distillation",
    "technique":        "Knowledge distillation",
    "backbone":         vars(scfg),
    "window":           cfg.window,
    "max_len":          cfg.max_len,
    "head_frac":        cfg.head_frac,
    "dev_macro_f1":     metrics["macro_f1"],
    "teacher_macro_f1": t_dev["macro_f1"],
    "kd_temp":          KD_TEMP,
    "kd_alpha":         KD_ALPHA,
    "history":          info.get("history"),
    "trainable":        int(trainable),
    "total":            int(total),
}, wpath)
print(f"\nstudent weights -> {wpath}  ({os.path.getsize(wpath)/1e6:.1f} MB)")

student: trainable 94,437,890 / 94,437,890

dev macro-F1 0.7052 | precision 0.7053 | recall 0.7052 | accuracy 0.7052
per-class F1: [0.7079, 0.7025]
recorded '06_distillation' -> /content/drive/MyDrive/cs781_phase4/results.json

student weights -> /content/drive/MyDrive/cs781_phase4/06_distillation.pt  (377.8 MB)


## 7 — Test submission

Writes `prediction.json` (the filename the competition expects) into `cs781_phase4/submissions/<experiment>/`.

The decision threshold is chosen on **dev** and applied unchanged to test — Phase 4 permits calibration and post-processing, and a skewed classifier usually leaves several macro-F1 points at the default 0.50.

In [ ]:
WRITE_SUBMISSION = True      # set False to skip (saves a few minutes)
USE_TUNED_THRESHOLD = True   # pick the decision threshold on dev, apply it to test

if WRITE_SUBMISSION:
    thr = 0.5
    if USE_TUNED_THRESHOLD:
        thr, f_at_thr, f_at_half = best_threshold(metrics["_logits"], metrics["_y"])
        print(f"dev macro-F1 at t=0.50 : {f_at_half:.4f}")
        print(f"dev macro-F1 at t={thr:.2f} : {f_at_thr:.4f}  ({f_at_thr-f_at_half:+.4f})")
        if f_at_thr - f_at_half < 0.002:
            thr = 0.5
            print("no real gain — falling back to 0.50")

    # the competition expects the file to be named prediction.json
    out = f"{OUT_DIR}/submissions/06_distillation/prediction.json"
    sub = write_submission(cfg, model, tokenizer, PAD_ID, out, threshold=thr)
    print("\nfirst rows:", sub[:3])
    print(f"\nupload this file: {out}")
else:
    print("skipped")

dev macro-F1 at t=0.50 : 0.7052
dev macro-F1 at t=0.49 : 0.7052  (+0.0000)
no real gain — falling back to 0.50


Phase_4_CJPE/Test/test.json: reconstructing file:   0%|          |  0.00B / 35.3MB            

Phase_4_CJPE/Test/test.json: downloading bytes:           |  0.00B            

test cases: 1517
  test 1517 docs in 8s
wrote /content/drive/MyDrive/cs781_phase4/submissions/06_distillation/prediction.json
  1517 rows | threshold 0.50 | predicted 0: 759  1: 758 (50.0% positive)

first rows: [{'id': '1986_397', 'pred': 1}, {'id': '1977_183', 'pred': 0}, {'id': '1993_98', 'pred': 1}]

upload this file: /content/drive/MyDrive/cs781_phase4/submissions/06_distillation/prediction.json


## 8 — Did this experiment do anything?

About **this** run only: whether it learned, whether it beats the trivial floors, whether its predictions are balanced, whether a different decision threshold would help, and where it fails.

In [ ]:
dev_rows = load_split(cfg, "dev")
thr = experiment_report(cfg, "06_distillation", info, metrics, dev_rows=dev_rows)

Phase_4_CJPE/Train/dev.json: reconstructing file:   0%|          |  0.00B / 45.0MB            

Phase_4_CJPE/Train/dev.json: downloading bytes:           |  0.00B            

EXPERIMENT REPORT — 06_distillation
  dev macro-F1 0.7052   precision 0.7053   recall 0.7052   accuracy 0.7052
  per-class F1 [0.7079, 0.7025]
  trained 4674 steps in 51.9 min

DID IT LEARN?  dev macro-F1 over training
----------------------------------------------------------
  step    400  0.5523  #
  step    800  0.6017  ###########
  step   1200  0.6093  #############
  step   1600  0.6563  ########################
  step   2000  0.6801  #############################
  step   2400  0.6719  ###########################
  step   2800  0.6808  #############################
  step   3200  0.6851  ##############################
  step   3600  0.6981  #################################
  step   4000  0.6974  #################################
  step   4400  0.7021  ##################################
  step   4674  0.7052  ###################################  <- best
----------------------------------------------------------
  first eval 0.5523  ->  best 0.7052   change +0.1529

IS IT BETTER

## 9 — Optional: all experiments side by side

Off by default — this notebook is about **this** experiment. Turn it on at the
end of the phase, when several runs are recorded, to write the comparison
section.

In [ ]:
SHOW_ALL = False      # set True once several experiments are recorded

if SHOW_ALL:
    compare_all(cfg)
else:
    print("off — set SHOW_ALL = True when you have several runs to compare")

off — set SHOW_ALL = True when you have several runs to compare
